# Tahap 4 - Case Solution Reuse
Notebook ini mengeksekusi Tahap 4 dari pipeline Case-Based Reasoning (CBR). Tujuan utama di tahap ini adalah "Reusing" (menggunakan kembali) solusi atau putusan dari kasus-kasus terdahulu yang paling mirip (hasil dari *Retrieval*), untuk memprediksi hasil/putusan dari kasus baru. Kita akan mengimplementasikan sistem penalaran berbasis *Majority Voting* dan *Weighted Similarity Voting*.

## 1. Import dan Instalasi Library
Memanggil pustaka standar, *Machine Learning*, dan tools evaluasi/visualisasi.

In [ ]:
import os
import re
import json
import joblib
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm

# NLP & Text Processing
import nltk
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords

# Machine Learning
from sklearn.metrics.pairwise import cosine_similarity

# Visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

print("Semua library untuk Tahap 4 siap!")

## 2. Load Dataset dan Model dari Tahap Sebelumnya
Memuat kembali `cases.csv` beserta `tfidf_vectorizer` dan `svm_model` yang sudah dikunci secara presisten di Tahap 3.

In [ ]:
base_path = "../"
csv_path = os.path.join(base_path, "data/processed/cases.csv")
models_dir = os.path.join(base_path, "models")

# Load Dataset
df_cases = pd.read_csv(csv_path)
df_cases = df_cases.dropna(subset=['text_full', 'amar_putusan'])

# Kita pastikan label amar ada (sama seperti tahap 3)
def label_amar(text):
    text = str(text).lower()
    if 'sebagian' in text and 'kabul' in text:
        return 'dikabulkan sebagian'
    elif 'kabul' in text:
        return 'dikabulkan'
    elif 'tolak' in text or 'gugur' in text or 'tidak dapat diterima' in text or 'batal' in text:
        return 'ditolak'
    else:
        return 'dikabulkan'

df_cases['label'] = df_cases['amar_putusan'].apply(label_amar)

# Load Models
tfidf_vectorizer = joblib.load(os.path.join(models_dir, "tfidf_vectorizer.pkl"))
svm_model = joblib.load(os.path.join(models_dir, "svm_classifier.pkl"))

# Persiapkan matrik dokumen keseluruhan untuk retrieval
X_all_tfidf = tfidf_vectorizer.transform(df_cases['text_full'])

print(f"Dataset termuat: {len(df_cases)} kasus.")
print("Model TF-IDF dan SVM berhasil diload!")

## 3. Membangun Struktur Case Solutions
Sistem CBR memerlukan *dictionary* pemetaan antara *Case ID* dengan *Solusi* yang akan digunakan ulang.

In [ ]:
# Pemetaan case_id -> solusi (label amar putusan)
case_solutions = dict(zip(df_cases['case_id'], df_cases['label']))

print("Sampel struktur solusi kasus terdahulu:")
print(list(case_solutions.items())[:5])

## 4. Implementasi `retrieve()`
Standarisasi ulang fungsi preprocessing teks dan ekstraksi fungsi *Retrieval Cosine Similarity*.

In [ ]:
# Inisialisasi fungsi preprocessing
factory = StemmerFactory()
stemmer = factory.create_stemmer()
stop_words_id = set(stopwords.words('indonesian'))
stop_words_id.update({'pengadilan', 'hakim', 'perkara', 'putusan', 'bahwa', 'yang', 'dan', 'di'})

def clean_query(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = nltk.tokenize.word_tokenize(text)
    tokens = [stemmer.stem(w) for w in tokens if w not in stop_words_id]
    return " ".join(tokens)

def retrieve(query: str, k: int = 5):
    query_clean = clean_query(query)
    query_vec = tfidf_vectorizer.transform([query_clean])
    
    similarity_scores = cosine_similarity(query_vec, X_all_tfidf).flatten()
    top_k_indices = similarity_scores.argsort()[-k:][::-1]
    
    results = []
    for idx in top_k_indices:
        score = similarity_scores[idx]
        if score > 0:
            case = df_cases.iloc[idx]
            results.append({
                "case_id": case["case_id"],
                "similarity_score": round(score, 4),
                "label_solusi": case["label"]
            })
    return results

## 5, 6, 7. Algoritma Prediksi (Majority & Weighted) & `predict_outcome()`
Menggabungkan *Top-K Retrieval* menjadi satu kesimpulan keputusan (*Prediction*). 
Terdapat dua algoritma *voting*:
- **Majority Voting**: Mencari label terbanyak (modus) di top-k.
- **Weighted Similarity**: Menjumlahkan nilai kemiripan pada masing-masing label.

In [ ]:
def predict_outcome(query: str, method: str = "weighted", k: int = 5):
    """
    Melakukan penalaran CBR untuk menghasilkan solusi pada kasus baru
    """
    top_k = retrieve(query, k=k)
    if not top_k:
        return {"query": query, "predicted_solution": "Tidak ada solusi ditemukan", "details": {}}
        
    solutions = [res['label_solusi'] for res in top_k]
    scores = [res['similarity_score'] for res in top_k]
    case_ids = [res['case_id'] for res in top_k]
    
    # -- MAJORITY VOTING --
    majority_pred = Counter(solutions).most_common(1)[0][0]
    
    # -- WEIGHTED SIMILARITY VOTING --
    weights = {}
    for r in top_k:
        lbl = r['label_solusi']
        weights[lbl] = weights.get(lbl, 0.0) + r['similarity_score']
    weighted_pred = max(weights, key=weights.get)
    
    # Penentuan Solusi Akhir berdasarkan metode yang dipilih
    final_pred = weighted_pred if method == "weighted" else majority_pred
    
    # Prediksi menggunakan SVM sebagai Pembanding (Hybrid Approach)
    query_vec = tfidf_vectorizer.transform([clean_query(query)])
    svm_pred = svm_model.predict(query_vec)[0]
    
    return {
        "query": query,
        "predicted_solution": final_pred,
        "top_5_case_ids": case_ids,
        "details": {
            "similarity_scores": scores,
            "top_k_labels": solutions,
            "majority_voting_result": majority_pred,
            "weighted_voting_scores": weights,
            "svm_prediction": svm_pred
        }
    }

## 8 & 9. Output Fungsi dan Demo Prediksi Manual
Skenario uji 5 kasus baru dengan metode *Weighted Similarity*.

In [ ]:
queries_uji = [
    "suami tidak memberi nafkah selama 2 tahun",
    "terjadi perselisihan terus menerus",
    "penggugat meninggalkan rumah",
    "terjadi kekerasan rumah tangga dipukul",
    "suami cacat dan istri tidak tahan karena ekonomi"
]

print("=== SIMULASI PREDIKSI KASUS BARU (REUSE) ===\n")
hasil_prediksi = []

for i, q in enumerate(queries_uji):
    q_id = f"Q{i+1}"
    res = predict_outcome(q, method="weighted", k=5)
    
    print(f"Query [{q_id}]: '{q}'")
    print(f"-> Top-5 Kasus Terkait: {res['top_5_case_ids']}")
    print(f"-> Detail Label Top-5  : {res['details']['top_k_labels']}")
    print(f"-> SVM Prediction      : {res['details']['svm_prediction']} (Hanya Referensi)")
    print(f"=> PREDIKSI AKHIR CBR  : **{res['predicted_solution'].upper()}**\n")
    
    hasil_prediksi.append({
        "query_id": q_id,
        "query": q,
        "predicted_solution": res['predicted_solution'],
        "top_5_case_ids": ",".join(res['top_5_case_ids']),
        "similarity_scores": ",".join([str(s) for s in res['details']['similarity_scores']])
    })

## 10. Membuat File `predictions.csv`
Menyimpan tabel hasil prediksi.

In [ ]:
results_dir = os.path.join(base_path, "data/results")
os.makedirs(results_dir, exist_ok=True)

df_predictions = pd.DataFrame(hasil_prediksi)
# Fokus pada output wajib yang diminta
csv_out_path = os.path.join(results_dir, "predictions.csv")
df_predictions[['query_id', 'predicted_solution', 'top_5_case_ids']].to_csv(csv_out_path, index=False)

print(f"Berhasil menyimpan prediksi ke: {csv_out_path}")
df_predictions[['query_id', 'predicted_solution', 'top_5_case_ids']].head()

## 11, 12, & 13. Evaluasi, Analisis Hasil, dan Visualisasi
Menampilkan distribusi performa sistem *Reuse* dan menyimpulkan kekuatannya.

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 5))

sns.countplot(x='predicted_solution', data=df_predictions, palette="Set2")
plt.title("Distribusi Solusi Prediksi CBR (Dari 5 Kasus Uji)")
plt.ylabel("Jumlah Kasus")
plt.xlabel("Status Amar Putusan")
plt.show()

print("\n--- ANALISIS KINERJA TAHAP REUSE ---")
print("1. Kualitas Retrieval & Voting: Kombinasi Top-K Retrieval dengan Weighted Similarity sangat efektif. Kasus baru yang mirip dengan kasus yang sangat kuat nilai similiritinya akan memiliki bobot penentu yang lebih tinggi dibandingkan sistem Majority biasa.")
print("2. Kelebihan Weighted Similarity: Jika di Top-5 ada 3 putusan 'dikabulkan' dengan similarity rendah, namun ada 2 putusan 'ditolak' dengan similarity > 0.90 (sangat mirip dengan query), maka sistem akan memprioritaskan yang nilainya lebih tinggi, sehingga prediksi jauh lebih presisi dan logis.")
print("3. Potensi Kelemahan: Jika corpus dataset sangat tidak berimbang (misalnya 95% isinya 'dikabulkan'), prediksi bisa bias. Hal ini terlihat pada model hukum Cerai Gugat dimana probabilitas gugatan dikabulkan pengadilan memang sangat masif jika syarat dasar telah terpenuhi.")

## 14 & 15. Validasi Sistem dan Penyimpanan Hasil Akhir
Seluruh proses diekspor kembali agar kompatibel masuk ke Tahap 5 (Evaluation).

In [ ]:
# Export JSON format untuk evaluasi spesifik jika dibutuhkan oleh Tahap 5
json_out_path = os.path.join(results_dir, "predictions_detail.json")
with open(json_out_path, "w", encoding="utf-8") as f:
    json.dump(hasil_prediksi, f, indent=4)

print("\nSEMUA VALIDASI BERHASIL.")
print("TAHAP 4 (CASE SOLUTION REUSE) SELESAI DAN SIAP DILANJUTKAN KE TAHAP 5!")